# 07 - Feature Engineering

This notebook engineers features from cleaned datasets for the machine learning model.

## What we'll do:
1. Load all cleaned interim datasets
2. Engineer meaningful features from the data
3. Create team statistics and form indicators
4. Prepare data for model training
5. Save processed training dataset

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, '../src')

from feature_engineering import (
    load_interim_datasets,
    engineer_features_optimized,
    calculate_head_to_head_features,
    preprocess_features,
    save_processed_data
)


## Step 1: Load Cleaned Datasets

In [ ]:
print("Loading cleaned interim datasets...")
results, elo, rankings, match_features = load_interim_datasets()

print(f"\nDatasets loaded:")
print(f"  • Results: {len(results)} matches")
print(f"  • Elo: {len(elo)} records")
print(f"  • Rankings: {len(rankings)} records")
print(f"  • Match Features: {len(match_features)} records")

In [ ]:
# Examine results structure
print("Results Dataset:")
print(results.columns.tolist())
display(results.head())

In [ ]:
# Examine Elo structure
print("Elo Ratings Dataset:")
print(elo.columns.tolist())
display(elo.head())

In [ ]:
# Examine Rankings structure
print("FIFA Rankings Dataset:")
print(rankings.columns.tolist())
display(rankings.head())

## Step 2: Prepare Team Statistics

In [ ]:
print('Calculating Head-to-Head historical records...')
results_h2h = calculate_head_to_head_features(results)
print(f'Head-to-head records calculated across {len(results_h2h)} matches.')
print(f"Matches with prior encounters: {(results_h2h['h2h_total_matches'] > 0).sum()}")
results_h2h[['date', 'home_team', 'away_team', 'h2h_total_matches', 'h2h_win_rate_diff']].tail(5)


## Step 3: Engineer Features

In [ ]:
print('Engineering features for all matches...\n')
engineered = engineer_features_optimized(results, elo, rankings, match_features)
print(f'Features engineered! Shape: {engineered.shape}')
print('\nFeatures created:')
for col in engineered.columns:
    print(f'  * {col}')


In [ ]:
# Display sample engineered features
display(engineered.head(10))

In [ ]:
# Show data types and missing values
print("Data Types and Missing Values:")
print(engineered.info())

## Step 4: Feature Analysis

In [ ]:
# Statistical summary of engineered features
print("Statistical Summary of Numerical Features:")
display(engineered.describe())

In [ ]:
# Distribution of match results
print("Match Result Distribution:")
print(engineered['match_result'].value_counts())
print("\nPercentages:")
print(engineered['match_result'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# Missing values summary
print("Missing Values Summary:")
missing = engineered.isnull().sum()
missing_pct = (engineered.isnull().sum() / len(engineered) * 100)
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Count': missing.values,
    'Missing %': missing_pct.values
})
display(missing_df[missing_df['Missing Count'] > 0])

## Step 5: Preprocess Features

In [ ]:
print("Preprocessing features...\n")
processed = preprocess_features(engineered)

print(f"Features preprocessed! Shape: {processed.shape}")
print(f"\nNew columns added:")
new_cols = set(processed.columns) - set(engineered.columns)
for col in new_cols:
    print(f"  • {col}")

In [ ]:
# Verify no missing values in critical columns
print("Missing values after preprocessing:")
missing_after = processed.isnull().sum()
print(missing_after[missing_after > 0])
print("\n✓ All missing values handled!" if missing_after.sum() == 0 else "⚠ Still have missing values")

In [ ]:
# Show sample of processed data
print("Sample of Processed Training Data:")
display(processed.head())

## Step 6: Feature Insights

In [ ]:
# Analyze correlation of features with match result
print('Correlation of Key Features with Goal Difference:')
key_features = [
    'elo_diff', 'rank_diff', 'home_advantage',
    'h2h_win_rate_diff', 'form_win_rate_diff', 'form_goal_diff',
    'overall_diff', 'attack_diff', 'defense_diff'
]
processed['goal_diff'] = processed['home_score'] - processed['away_score']
available_keys = [f for f in key_features if f in processed.columns]
correlations = processed[available_keys + ['goal_diff']].corr()['goal_diff'].sort_values(ascending=False)
print(correlations)


## Step 7: Save Processed Data

In [ ]:
# Save the processed training dataset
save_processed_data(processed, 'training_data.csv')

print(f"\n✓ Training dataset saved successfully!")
print(f"\nDataset Summary:")
print(f"  • Total Matches: {len(processed)}")
print(f"  • Total Features: {len(processed.columns)}")
print(f"  • Date Range: {processed['date'].min()} to {processed['date'].max()}")
print(f"\nFeature Categories:")
print(f"  • Elo-based: elo_diff, home_elo, away_elo")
print(f"  • Ranking-based: rank_diff, home_rank, away_rank")
print(f"  • Form-based: win_rate_diff, home_win_rate, away_win_rate")
print(f"  • Goals-based: home_goals_per_match, away_goals_per_match, home_goals_conceded_per_match, away_goals_conceded_per_match")
print(f"  • Head-to-head: h2h_matches, home_h2h_wins, away_h2h_wins")
print(f"  • Target: match_result_encoded (0=Home Win, 1=Draw, 2=Away Win)")

In [ ]:
# Final summary
print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE!")
print("="*60)
print(f"\n✓ Successfully engineered {len(processed)} match records")
print(f"✓ Created {len(processed.columns)} features")
print(f"✓ Saved to: data/processed/training_data.csv")
print(f"\nNext Step: Model Training (Notebook 08)")